In [ ]:
# INSTALL
!pip install openai jsonschema joblib

import os
import json
import re
import joblib
from jsonschema import validate, ValidationError
from openai import OpenAI

# API KEY (put your key)
os.environ["LLM_API_KEY"] =input("paste your OpenRouter key:")

client = OpenAI(
    api_key=os.getenv("LLM_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

# LOAD MODEL
model = joblib.load("best_model.pkl")

# LLM CALL FUNCTION
def call_llm(system_prompt, user_prompt, temperature=0):
    response = client.chat.completions.create(
        model="openai/gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=temperature
    )
    return response.choices[0].message.content

# TEST API
print(call_llm("Reply only hello", "test"))

# SYSTEM PROMPT
system_prompt = """
You are an AI model explanation assistant.
Return ONLY valid JSON.

Schema:
{
  "prediction_label": "string",
  "confidence_level": "low|medium|high",
  "top_reason": "string",
  "second_reason": "string",
  "next_step": "string"
}
"""

# JSON SCHEMA
schema = {
    "type": "object",
    "properties": {
        "prediction_label": {"type": "string"},
        "confidence_level": {"type": "string"},
        "top_reason": {"type": "string"},
        "second_reason": {"type": "string"},
        "next_step": {"type": "string"}
    },
    "required": [
        "prediction_label",
        "confidence_level",
        "top_reason",
        "second_reason",
        "next_step"
    ]
}

# PII CHECK
def has_pii(text):
    email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]+'
    phone_pattern = r'\b\d{10}\b'
    return bool(re.search(email_pattern, text) or re.search(phone_pattern, text))

# SAMPLE INPUTS
inputs = [
    {"Age": 25, "Balance": 5000, "Tenure": 2},
    {"Age": 45, "Balance": 20000, "Tenure": 8},
    {"Age": 60, "Balance": 1000, "Tenure": 1}
]

for data in inputs:

    if has_pii(str(data)):
        print("Blocked due to PII")
        continue

    # Fake prediction (replace if model needs full features)
    prediction = model.predict([[1]*model.n_features_in_])[0]
    probability = model.predict_proba([[1]*model.n_features_in_])[0].max()

    user_prompt = f"""
    Features: {data}
    Predicted class: {prediction}
    Probability: {probability}

    Explain in JSON format.
    """

    raw_output = call_llm(system_prompt, user_prompt)

    try:
        parsed = json.loads(raw_output.strip())
        validate(instance=parsed, schema=schema)
        print("VALID JSON")
        print(parsed)
    except Exception as e:
        print("FAILED:", e)

# TEMPERATURE COMPARISON
print("\nTemperature 0:")
print(call_llm(system_prompt, user_prompt, temperature=0))

print("\nTemperature 0.7:")
print(call_llm(system_prompt, user_prompt, temperature=0.7))